In [1]:
import pandas as pd

In [51]:
student_observations = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Student_Observations')
class_plan = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Class_Plan',index_col='class_num')
kc_nodes = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='KC_Nodes')
hwk_items = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Homework_Items',index_col='class_num')

Need the following columns :
| DataShop col | Data col | File |
|--------------|----------|------|
|Anon Student ID | student_id | student_observations |
|Seesion ID | assignment_id | student_observations |
|Time| class_date | class_plan |
|Level | reporting_group | kc_nodes |
|Problem name | source_question | student_observations |
|Step name | observation_id | student_observations |
|Outcome | score | student_observations |
|KC | primary_kc_id | student_observations |
|class| class_num | student_observations |

## Preprocessing

### Student Observations

In [5]:
student_observations.head()

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.0,1,0.0,C,E,NaN
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,100.0,E,E,NaN
2,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,0.0,1,0.0,C,D,NaN
3,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100.0,E,E,NaN
4,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100.0,B,B,NaN


Replace score 0/1 scores with correct, incorrect

In [16]:
scores = [
    (student_observations['score']==student_observations['max_score'], 'CORRECT'),
    (student_observations['score']!=student_observations['max_score'], 'INCORRECT')
]
student_observations['Outcome']=student_observations['score'].case_when(scores)
student_observations[['student_id','assignment_id','source_question','observation_id','Outcome','primary_kc_id','class_num']].rename(columns={'student_id':'Anon Student Id',
                                                                                                                                              'assignment_id':'Session Id',
                                                                                                                                              'source_question':'Problem name',
                                                                                                                                              'observation_id':'Step name',
                                                                                                                                              'primary_kc_id' : 'KC',
                                                                                                                                              'class_num':'Class'})

,Anon Student Id,Session Id,Problem name,Step name,Outcome,KC,Class
0,S001,HW1,PCA Q01,HW1_PCA_Q01,INCORRECT,KC.U1.02.observational_unit_variable,1
1,S001,HW1,PCA Q02,HW1_PCA_Q02,CORRECT,KC.U1.03.variable_type_cat_quant,1
2,S001,HW1,PCA Q03,HW1_PCA_Q03,INCORRECT,KC.U1.03.variable_type_cat_quant,1
3,S001,HW1,PCA Q04,HW1_PCA_Q04,CORRECT,KC.U1.05.categorical_freq_relative,1
4,S001,HW1,PCA Q05,HW1_PCA_Q05,CORRECT,KC.U1.05.categorical_freq_relative,1
...,...,...,...,...,...,...,...
21803,S025,MF2,FRQ 6,MF2_FRQ6A,INCORRECT,KC.U1.09.quantitative_display_reading,27
21804,S025,MF2,FRQ 6,MF2_FRQ6B,CORRECT,KC.U7.12.mean_hypotheses,27
21805,S025,MF2,FRQ 6,MF2_FRQ6C,INCORRECT,KC.U7.13.t_test_conditions,27
21806,S025,MF2,FRQ 6,MF2_FRQ6D,CORRECT,KC.U7.14.t_test_statistic_mean,27


### Class plan

In [52]:
class_plan=class_plan[['class_date']]
class_plan.head()

,class_date
class_num,
1,2025-09-05
2,2025-09-12
3,2025-09-19
4,2025-09-26
5,2025-10-03


In [53]:
hwk_items=hwk_items[['sequence','observation_id']]

In [54]:
hwk_items.join(class_plan)

,sequence,observation_id,class_date
class_num,,,
1,1,HW1_PCA_Q01,2025-09-05
1,2,HW1_PCA_Q02,2025-09-05
1,3,HW1_PCA_Q03,2025-09-05
1,4,HW1_PCA_Q04,2025-09-05
1,5,HW1_PCA_Q05,2025-09-05
...,...,...,...
27,56,MF2_FRQ6A,2026-03-13
27,57,MF2_FRQ6B,2026-03-13
27,58,MF2_FRQ6C,2026-03-13
